# Formula 1 Race Strategy Analysis — COTA (2019–2025)

## Dataset Overview

This notebook explores historical Formula 1 race strategy data for the **United States Grand Prix** at the Circuit of the Americas (COTA) in Austin, Texas. The data spans the 2019, 2021, 2022, 2023, 2024, and 2025 seasons (2020 was cancelled due to COVID-19).

**Source:** The data is extracted from the official F1 timing system via the [FastF1](https://docs.fastf1.dev/) Python library, then processed through a custom pipeline that reconstructs stint-level strategy records, pit stop events, and race control messages for each season.

**Primary dataset (`strategy_raw.csv`)** — one row per driver per stint per race year:

| Attribute | Type | Description |
|-----------|------|-------------|
| `year` | Ordinal | Season (2019–2025) |
| `driver` | Nominal | Three-letter driver name abbreviation |
| `team` | Nominal | Constructor name (normalized to 2025 identity as some teams have changed names frequently) |
| `grid_position` | Quantitative | Starting position on the grid |
| `finish_position` | Quantitative | Final classified position |
| `strategy` | Nominal | Compound sequence code (e.g., `M-H` for Medium→Hard) |
| `num_stops` | Ordinal | Total pit stops made |
| `stint_num` | Ordinal | Stint index within the driver's race |
| `compound` | Nominal | Tire compound for this stint (SOFT, MEDIUM, HARD, etc.) |
| `stint_start_lap` | Quantitative | First lap of the stint |
| `stint_end_lap` | Quantitative | Last lap of the stint |
| `stint_length` | Quantitative | Duration of the stint in laps |

**Secondary dataset (`pit_stops_raw.csv`)** — one row per pit stop:

| Attribute | Type | Description |
|-----------|------|-------------|
| `year` | Ordinal | Season |
| `driver` | Nominal | Driver name abbreviation |
| `team` | Nominal | Constructor |
| `pit_lap` | Quantitative | Lap number of the pit stop |
| `compound_before` / `compound_after` | Nominal | Tire compounds swapped |
| `position_before` / `position_after` | Quantitative | Track position pre/post stop |
| `position_delta` | Quantitative | Positions gained (positive) or lost (negative) |

### Goals

1. **How has the dominant strategy evolved across seasons?** Has COTA shifted from a 1-stop to a 2-stop race, or vice versa?
2. **Does grid position constrain strategy choice?** Do frontrunners and midfield/backmarker drivers make systematically different compound choices?
3. **Which strategies correlate with positions gained?** Are aggressive multi-stop strategies rewarded at this circuit?
4. **How do pit stop windows cluster, and has that window shifted over time?**
5. **Do teams have distinctive strategic signatures?** Can we see team philosophy differences in compound and timing choices?

In [2]:
import pandas as pd
import altair as alt

alt.data_transformers.disable_max_rows()
OUTPUT_DIR = "."

strategy_raw = pd.read_csv(f"{OUTPUT_DIR}/strategy_raw.csv")
pit_stops     = pd.read_csv(f"{OUTPUT_DIR}/pit_stops_raw.csv")


print(f"strategy_raw: {strategy_raw.shape[0]} rows × {strategy_raw.shape[1]} columns")
print(f"pit_stops:    {pit_stops.shape[0]} rows × {pit_stops.shape[1]} columns")
strategy_raw.head()

strategy_raw: 261 rows × 12 columns
pit_stops:    200 rows × 9 columns


,year,driver,team,grid_position,finish_position,strategy,num_stops,stint_num,compound,stint_start_lap,stint_end_lap,stint_length
0,2019,GAS,RB,10,16,S-H-S,2,1,SOFT,1,19,19
1,2019,GAS,RB,10,16,S-H-S,2,2,HARD,20,53,34
2,2019,GAS,RB,10,16,S-H-S,2,3,SOFT,54,54,1
3,2019,PER,Aston Martin,0,10,M-H,1,1,MEDIUM,1,24,24
4,2019,PER,Aston Martin,0,10,M-H,1,2,HARD,25,55,31


In [3]:

driver_races = (
    strategy_raw
    .groupby(["year", "driver"], as_index=False)
    .first()
    .assign(positions_gained=lambda df: df["grid_position"] - df["finish_position"])
)

driver_races["year"] = driver_races["year"].astype(str)
strategy_raw["year"] = strategy_raw["year"].astype(str)
pit_stops["year"] = pit_stops["year"].astype(str)


compound_scale = alt.Scale(
    domain=["SOFT", "MEDIUM", "HARD", "INTERMEDIATE", "WET"],
    range=["#FF3333", "#FFC300", "#AAAAAA", "#39B54A", "#0072C6"]
)

print(f"Unique years: {sorted(driver_races['year'].unique())}")
print(f"Drivers per year: {driver_races.groupby('year')['driver'].nunique().to_dict()}")

Unique years: ['2019', '2021', '2022', '2023', '2024', '2025']
Drivers per year: {'2019': 19, '2021': 20, '2022': 18, '2023': 18, '2024': 19, '2025': 19}


---
## 1 — Strategy Distribution Across Seasons

The first question is whether COTA's dominant strategy has been stable or shifting. A normalized stacked bar chart by year shows the proportion of 1-stop, 2-stop, and 3-stop races.

In [4]:
stop_counts = (
    driver_races
    .groupby(["year", "num_stops"], as_index=False)
    .agg(count=("driver", "count"))
)

stop_color_scale = alt.Scale(
    domain=[1, 2, 3],
    range=["#2066a8", "#d47264", "#ae282c"]
)

alt.Chart(stop_counts).mark_bar().encode(
    x=alt.X("year:N", title="Season"),
    y=alt.Y("count:Q", stack="normalize", title="Proportion of Field",
            axis=alt.Axis(format="%")),
    color=alt.Color("num_stops:N", title="Stops",
                    scale=stop_color_scale,
                    legend=alt.Legend(orient="right")),
    tooltip=["year", "num_stops", "count"]
).properties(
    title="COTA — Pit Stop Count Distribution by Season",
    width=480,
    height=300
)

alt.Chart(...)

---
## 2 — Grid Position vs. Strategy Choice

A normalized horizontal stacked bar chart showing the proportion of drivers who chose each stop count (1-stop, 2-stop, 3-stop) within each grid position group. The grid groups are binned into frontrunners (P1–P3), midfield tiers, and backmarkers (P16+).

In [5]:
driver_races["grid_group"] = pd.cut(
    driver_races["grid_position"],
    bins=[0, 3, 6, 10, 15, 25],
    labels=["P1–P3", "P4–P6", "P7–P10", "P11–P15", "P16+"]
)

grid_stop = (
    driver_races
    .dropna(subset=["grid_group"])
    .groupby(["grid_group", "num_stops"], observed=True)
    .size()
    .reset_index(name="count")
)

stop_color_scale = alt.Scale(
    domain=[3, 2, 1],
    range=["#ae282c", "#d47264", "#2066a8"]
)

alt.Chart(grid_stop).mark_bar().encode(
    y=alt.Y("grid_group:N", title="Grid Position Group",
            sort=["P1–P3", "P4–P6", "P7–P10", "P11–P15", "P16+"]),
    x=alt.X("count:Q", stack="normalize", title="Proportion of Drivers",
            axis=alt.Axis(format="%")),
    color=alt.Color("num_stops:N", title="Stops",
                    scale=stop_color_scale,
                    legend=alt.Legend(orient="right")),
    order=alt.Order("num_stops:Q", sort="descending"),
    tooltip=["grid_group", "num_stops", "count"]
).properties(
    title="Strategy Choice by Grid Position Group",
    width=450,
    height=250
)

alt.Chart(...)

---
## 3 — Stint Structure: Compound Usage Across the Race (Interactive)

This Gantt-style chart shows every stint for every driver in a selected year, encoding compound by color and stint duration by bar length. Use the dropdown under the chart to switch between seasons.

In [6]:
year_dropdown = alt.binding_select(
    options=sorted(strategy_raw["year"].unique().tolist()),
    name="Season: "
)
year_select = alt.selection_point(
    fields=["year"],
    bind=year_dropdown,
    value=sorted(strategy_raw["year"].unique().tolist())[0]
)

# Sort drivers by finish position within each year
stint_data = strategy_raw.merge(
    driver_races[["year", "driver", "finish_position"]],
    on=["year", "driver"],
    how="left",
    suffixes=("", "_race")
)

alt.Chart(stint_data).mark_bar(height=12).encode(
    x=alt.X("stint_start_lap:Q", title="Lap"),
    x2="stint_end_lap:Q",
    y=alt.Y("driver:N", title="Driver",
            sort=alt.EncodingSortField(field="finish_position", op="min")),
    color=alt.Color("compound:N", title="Compound", scale=compound_scale),
    tooltip=["driver", "team", "compound", "stint_start_lap", "stint_end_lap", "stint_length"]
).add_params(
    year_select
).transform_filter(
    year_select
).properties(
    title="Stint Structure by Driver (select season)",
    width=600,
    height=450
)

alt.Chart(...)

---
## 4 — Pit Stop Windows Over Time (Interactive)

A two-panel chart showing pit stop timing across seasons. The upper strip plot places each individual pit stop as a dot on the lap axis, grouped by year with a unique color per season. The lower panel draws an aggregate kernel density estimate to highlight where stops cluster most heavily. The two panels are linked by a shared selection: clicking a year row in the strip plot grays out the other seasons and redraws the density curve using only that year's stops, letting the reader compare how the preferred pit window has shifted from season to season. Clicking empty space resets to the full aggregate view.

In [21]:
year_list = sorted(pit_stops["year"].unique().tolist())
year_df = pd.DataFrame({"year": year_list})

year_select = alt.selection_point(fields=["year"])

backgrounds = alt.Chart(year_df).mark_rect(opacity=0).encode(
    y=alt.Y("year:N", sort=year_list)
).add_params(
    year_select
).properties(
    width=600,
    height=200
)

dots = alt.Chart(pit_stops).mark_circle(size=55, opacity=0.8).encode(
    x=alt.X("pit_lap:Q", title=""),
    y=alt.Y("year:N", title="Season", sort=year_list),
    color=alt.condition(
        year_select,
        alt.Color("compound_before:N", title="Compound",
                  scale=compound_scale,
                  legend=alt.Legend(orient="right")),
        alt.value("#d9d9d9")
    ),
    tooltip=["year", "driver", "team", "pit_lap",
             "compound_before", "compound_after"]
)

strip = (backgrounds + dots).properties(
    title=alt.TitleParams(
    "United States Grand Prix — Pit Stop Windows (2019, 2021-2025)",
    subtitle="Click a year row to filter the density curve, double click to clear"
),
    width=600,
    height=200
)

density = alt.Chart(pit_stops).transform_filter(
    year_select
).transform_density(
    "pit_lap",
    as_=["pit_lap", "density"],
    groupby=["compound_before"],
    bandwidth=3
).mark_area(
    opacity=0.3,
    line={"strokeWidth": 2}
).encode(
    x=alt.X("pit_lap:Q", title="Lap Number"),
    y=alt.Y("density:Q", title="", axis=None),
    color=alt.Color("compound_before:N", scale=compound_scale, legend=None)
).properties(
    width=600,
    height=100
)

(strip & density).resolve_scale(x="shared")

chart = (strip & density).resolve_scale(x="shared")
chart.save("cota_pit_windows.html")
chart

alt.VConcatChart(...)

---
## 5 — Positions Gained/Lost at Pit Stops (Interactive)

A linked two-panel view of pit stop position impact. The left histogram shows the distribution of positions gained or lost across all pit stops, while the right bar chart shows the average position change grouped by the compound fitted during the stop. The two panels are linked by a shared selection: clicking a compound on either panel highlights the corresponding data in the other.

In [8]:
pit_clean = pit_stops.dropna(subset=["position_delta"]).copy()
pit_clean["position_delta"] = pd.to_numeric(pit_clean["position_delta"], errors="coerce")

compound_select = alt.selection_point(fields=["compound_after"])

hist = alt.Chart(pit_clean).mark_bar().encode(
    x=alt.X("position_delta:Q", bin=alt.Bin(step=1),
            title="Positions Gained (+) / Lost (−)"),
    y=alt.Y("count()", title="Number of Pit Stops"),
    color=alt.condition(
        compound_select,
        alt.Color("compound_after:N", scale=compound_scale, legend=None),
        alt.value("#d9d9d9")
    ),
    tooltip=["compound_after:N", "count()"]
).properties(
    title="Distribution of Position Change at Pit Stops",
    width=350,
    height=300
).add_params(compound_select)

compound_avg = pit_clean.groupby("compound_after", as_index=False).agg(
    avg_delta=("position_delta", "mean")
).sort_values("avg_delta", ascending=False)

bars = alt.Chart(compound_avg).mark_bar().encode(
    x=alt.X("compound_after:N", title="Compound Fitted",
            sort=compound_avg["compound_after"].tolist()),
    y=alt.Y("avg_delta:Q", title="Avg Positions Gained"),
    color=alt.condition(
        compound_select,
        alt.Color("compound_after:N", scale=compound_scale, legend=None),
        alt.value("#d9d9d9")
    ),
    tooltip=["compound_after", alt.Tooltip("avg_delta:Q", format=".1f")]
).add_params(compound_select)

zero_line = alt.Chart(pd.DataFrame({"y": [0]})).mark_rule(
    color="gray", strokeDash=[4, 4]
).encode(y="y:Q")

compound_chart = (bars + zero_line).properties(
    title="Avg Position Change by Compound (click to filter)",
    width=250,
    height=300
)

hist | compound_chart


alt.HConcatChart(...)

---
## 6 — Strategy vs. Outcome: Positions Gained by Strategy Type

A box plot showing the distribution of positions gained (grid minus finish) for each strategy string, filtered to strategies used at least twice. Strategies above zero tend to yield forward progress.

In [9]:
# Filter to strategies with at least 2 uses
strat_counts = driver_races["strategy"].value_counts()
common_strats = strat_counts[strat_counts >= 2].index.tolist()
dr_common = driver_races[driver_races["strategy"].isin(common_strats)].copy()

alt.Chart(dr_common).mark_boxplot(extent="min-max").encode(
    x=alt.X("positions_gained:Q", title="Positions Gained (Grid − Finish)"),
    y=alt.Y("strategy:N", title="Strategy",
            sort=alt.EncodingSortField(field="positions_gained", op="median",
                                       order="descending")),
    color=alt.Color("num_stops:N", title="Stops",
                    scale=stop_color_scale)
).properties(
    title="Positions Gained by Strategy (min 2 uses)",
    width=500,
    height=350
)

alt.Chart(...)

---
## 7 — Team Strategy Signatures (Interactive)

An interactive chart showing each team's average number of stops alongside their compound preferences. Click a team to highlight its pit stops in the linked chart below.

In [10]:
team_selection = alt.selection_point(fields=["team"])

team_avg = (
    driver_races
    .groupby("team", as_index=False)
    .agg(
        avg_stops=("num_stops", "mean"),
        races=("driver", "count")
    )
    .sort_values("avg_stops")
)

bars = alt.Chart(team_avg).mark_bar().encode(
    x=alt.X("avg_stops:Q", title="Average Stops"),
    y=alt.Y("team:N", title="Team", sort=alt.EncodingSortField(field="avg_stops")),
    color=alt.condition(
        team_selection,
        alt.value("#2066a8"),
        alt.value("#d9d9d9")
    ),
    tooltip=["team", "avg_stops", "races"]
).add_params(
    team_selection
).properties(
    title="Average Stops by Team",
    width=400,
    height=300
)

# Linked: compound usage for selected team
compound_usage = (
    strategy_raw
    .groupby(["team", "compound"], as_index=False)
    .agg(stint_count=("stint_num", "count"))
)

compound_bars = alt.Chart(compound_usage).mark_bar().encode(
    x=alt.X("stint_count:Q", title="Number of Stints"),
    y=alt.Y("compound:N", title="Compound",
            sort=["SOFT", "MEDIUM", "HARD", "INTERMEDIATE", "WET"]),
    color=alt.Color("compound:N", scale=compound_scale, legend=None),
    tooltip=["team", "compound", "stint_count"]
).transform_filter(
    team_selection
).properties(
    title="Compound Usage (click a team bar in the Avg Stops by Team to filter)",
    width=400,
    height=300
)

bars | compound_bars

alt.HConcatChart(...)

---
## 8 — Stint Length by Compound (Interactive)

An interval selection lets you brush a range of stint lengths on the histogram and see the corresponding stints highlighted in the scatter plot of stint start lap vs. length. Click and drag a selection on the bar chart to select range. Drag selection to brush. Double click to clear selection.

In [11]:
brush = alt.selection_interval(encodings=["x"])

hist = alt.Chart(strategy_raw).mark_bar(opacity=0.8).encode(
    x=alt.X("stint_length:Q", bin=alt.Bin(step=2), title="Stint Length (laps)"),
    y=alt.Y("count()", title="Count"),
    color=alt.condition(
        brush,
        alt.Color("compound:N", scale=compound_scale),
        alt.value("lightgray")
    )
).add_params(
    brush
).properties(
    title="Stint Length Distribution (brush to filter)",
    width=600,
    height=200
)

scatter = alt.Chart(strategy_raw).mark_circle(size=40, opacity=0.6).encode(
    x=alt.X("stint_start_lap:Q", title="Stint Start Lap"),
    y=alt.Y("stint_length:Q", title="Stint Length"),
    color=alt.condition(
        brush,
        alt.Color("compound:N", scale=compound_scale),
        alt.value("lightgray")
    ),
    tooltip=["year", "driver", "compound", "stint_start_lap", "stint_length"]
).properties(
    title="Stint Start Lap vs. Length",
    width=600,
    height=250
)

hist & scatter

alt.VConcatChart(...)